# v25 — Surgical Fine-Tuning with an Explicit Retention Constraint

**Question.** Can target science-reasoning improve while held-out code behaviour remains within a predeclared tolerance?

This notebook does not claim that an activation covariance is a Fisher metric or that any initializer guarantees invariance. Its primary intervention is a direct retained-behaviour KL penalty during LoRA fine-tuning.

## Locked protocol

- **Target train:** GPQA Main calibration split plus SciQ.
- **Target final:** GPQA Diamond. It is not used for training, covariance construction, or selection.
- **Retention train:** a deterministic HumanEval calibration split.
- **Retention final:** a disjoint HumanEval split.
- **Fair arms:** standard LoRA; standard LoRA plus retention KL; activation-whitened initialization plus the same KL (explicitly exploratory). All arms train both LoRA A and B at the same rank, layers, steps, and learning rate.
- **Success:** a positive lower 95% CI for GPQA gain **and** a held-out code-NLL upper 95% CI no larger than the declared tolerance.

Full generated answers are saved, so scoring can be independently audited. A code NLL change is signed: positive is degradation and negative is improvement.

In [ ]:
from pathlib import Path
import sys

CWD = Path.cwd().resolve()
WORK = CWD if (CWD / 'laguna').is_dir() else CWD.parent
MODULE_DIR = WORK / 'laguna'
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from v25_surgical_constrained_lora import make_config, run_v25

WORK

## Configuration

Set these values before the first run and archive the resulting `protocol_config.json`. Do not tune them using the final GPQA Diamond or final HumanEval partitions.

In [ ]:
config = make_config(
    workdir=WORK,
    retention_lambda=1.0,        # Choose using development data only; lock before final reads.
    control_nll_tolerance=0.02, # Positive held-out NLL increase allowed by the protocol.
    seeds=[107, 211, 503, 719, 941],
    run_activation_whitened_arm=True,
)
config

## Run

This loads Laguna, creates the locked split manifest, runs all common-seed arms, and writes raw generations, per-task code NLLs, initial adapter norms, train histories, and hierarchical bootstrap summaries under `v25_artifacts/`.

In [ ]:
summary = run_v25(config)
summary

## Interpretation rule

Do not call an arm surgical because it has a high mean score, because its activations look separated, or because its code metric merely moved. It qualifies only when the locked final summary reports both `target_success=True` and `control_retained=True`. This test establishes code retention only; add disjoint instruction-following and factual-quality suites before claiming broad quality preservation.